# DWH Laden
Dit notebook laadt de data vanuit het SDM naar het DWH sterschema.

In [1]:
import sqlite3
import pandas as pd
from datetime import datetime

SDM_PATH = "../data/sdm_deproject.db"
DWH_PATH = "../data/dwh_deproject.db"

sdm_conn = sqlite3.connect(SDM_PATH)
dwh_conn = sqlite3.connect(DWH_PATH)

print("Verbindingen gemaakt.")

Verbindingen gemaakt.


### 1. Dim_Product

In [2]:
query = """
SELECT 
    p.product_number AS product_key,
    p.product_name,
    pt.product_type_en AS product_type,
    pl.product_line_en AS product_line,
    p.introduction_date
FROM product p
JOIN product_type pt ON p.product_type_code = pt.product_type_code
JOIN product_line pl ON pt.product_line_code = pl.product_line_code
"""
dim_product = pd.read_sql(query, sdm_conn)
dim_product.to_sql("Dim_Product", dwh_conn, if_exists="append", index=False)
print(f"Dim_Product geladen: {len(dim_product)} records")

Dim_Product geladen: 115 records


### 2. Dim_Staff

In [3]:
query = """
SELECT 
    s.sales_staff_code AS staff_key,
    s.first_name || ' ' || s.last_name AS staff_full_name,
    s.position_en AS position,
    b.city AS work_location
FROM sales_staff s
JOIN sales_branch b ON s.sales_branch_code = b.sales_branch_code
"""
dim_staff = pd.read_sql(query, sdm_conn)
dim_staff.to_sql("Dim_Staff", dwh_conn, if_exists="append", index=False)
print(f"Dim_Staff geladen: {len(dim_staff)} records")

Dim_Staff geladen: 96 records


### 3. Dim_Customer

In [4]:
query = """
SELECT DISTINCT
    c.CUSTOMER_CODE AS customer_key,
    c.COMPANY_NAME,
    ct.CUSTOMER_TYPE_EN AS customer_segment,
    co.country,
    rs.region
FROM customer c
JOIN customer_type ct ON c.CUSTOMER_TYPE_CODE = ct.CUSTOMER_TYPE_CODE
LEFT JOIN retailer_site rs ON c.CUSTOMER_CODE = rs.retailer_code
LEFT JOIN country co ON rs.country_code = co.country_code
GROUP BY c.CUSTOMER_CODE
"""
dim_customer = pd.read_sql(query, sdm_conn)
dim_customer.to_sql("Dim_Customer", dwh_conn, if_exists="append", index=False)
print(f"Dim_Customer geladen: {len(dim_customer)} records")

Dim_Customer geladen: 109 records


### 4. Dim_Date

In [5]:
# Verzamel alle datums uit de feiten om de Dim_Date te vullen
dates = pd.read_sql("SELECT order_date FROM order_header", sdm_conn)['order_date']
dates = pd.to_datetime(dates).unique()

date_df = pd.DataFrame({'date': dates})
date_df['date_key'] = date_df['date'].dt.strftime('%Y%m%d').astype(int)
date_df['year'] = date_df['date'].dt.year
date_df['quarter'] = date_df['date'].dt.quarter
date_df['month_name'] = date_df['date'].dt.month_name()
date_df['month_number'] = date_df['date'].dt.month
date_df['year_month'] = date_df['date'].dt.strftime('%Y-%m')

date_df.to_sql("Dim_Date", dwh_conn, if_exists="append", index=False)
print(f"Dim_Date geladen: {len(date_df)} records")

C:\Users\xande\AppData\Local\Temp\ipykernel_15656\3781189524.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(dates).unique()


Dim_Date geladen: 911 records


### 5. Fact_Sales_Actuals

In [6]:
query = """
SELECT 
    CAST(strftime('%Y%m%d', h.order_date) AS INTEGER) AS date_key,
    d.product_number AS product_key,
    rs.retailer_code AS customer_key,
    h.sales_staff_code AS staff_key,
    d.quantity,
    d.unit_cost,
    d.unit_price,
    (d.quantity * d.unit_sale_price) AS revenue,
    ((d.unit_sale_price - d.unit_cost) * d.quantity) AS gross_Margin
FROM order_details d
JOIN order_header h ON d.order_number = h.order_number
JOIN retailer_site rs ON h.retailer_site_code = rs.retailer_site_code
"""
fact_sales = pd.read_sql(query, sdm_conn)
fact_sales.to_sql("Fact_Sales_Actuals", dwh_conn, if_exists="append", index=False)
print(f"Fact_Sales_Actuals geladen: {len(fact_sales)} records")

Fact_Sales_Actuals geladen: 40990 records


### 6. Fact_Sales_Targets

In [7]:
# Sales targets hebben jaar en periode (maand). We koppelen dit aan de eerste van de maand.
query = """
SELECT 
    CAST(SALES_YEAR || printf('%02d', SALES_PERIOD) || '01' AS INTEGER) AS date_key,
    PRODUCT_NUMBER AS product_key,
    RETAILER_CODE AS customer_key,
    SALES_STAFF_CODE AS staff_key,
    SALES_TARGET
FROM sales_target
"""
fact_targets = pd.read_sql(query, sdm_conn)

# Zorg dat deze datums ook in Dim_Date staan indien nog niet aanwezig
target_dates = pd.to_datetime(fact_targets['date_key'].astype(str), format='%Y%m%d').unique()
existing_dates = pd.read_sql("SELECT date FROM Dim_Date", dwh_conn)['date']
existing_dates = pd.to_datetime(existing_dates)

new_dates = [d for d in target_dates if d not in existing_dates.values]
if new_dates:
    new_date_df = pd.DataFrame({'date': new_dates})
    new_date_df['date_key'] = new_date_df['date'].dt.strftime('%Y%m%d').astype(int)
    new_date_df['year'] = new_date_df['date'].dt.year
    new_date_df['quarter'] = new_date_df['date'].dt.quarter
    new_date_df['month_name'] = new_date_df['date'].dt.month_name()
    new_date_df['month_number'] = new_date_df['date'].dt.month
    new_date_df['year_month'] = new_date_df['date'].dt.strftime('%Y-%m')
    new_date_df.to_sql("Dim_Date", dwh_conn, if_exists="append", index=False)

fact_targets.to_sql("Fact_Sales_Targets", dwh_conn, if_exists="append", index=False)
print(f"Fact_Sales_Targets geladen: {len(fact_targets)} records")

Fact_Sales_Targets geladen: 39530 records


### 7. Fact_Product_Forecast

In [8]:
query = """
SELECT 
    CAST(YEAR || printf('%02d', MONTH) || '01' AS INTEGER) AS date_key,
    PRODUCT_NUMBER,
    EXPECTED_VOLUME
FROM product_forecast
"""
fact_forecast = pd.read_sql(query, sdm_conn)

# Zorg dat deze datums ook in Dim_Date staan
forecast_dates = pd.to_datetime(fact_forecast['date_key'].astype(str), format='%Y%m%d').unique()
existing_dates = pd.read_sql("SELECT date FROM Dim_Date", dwh_conn)['date']
existing_dates = pd.to_datetime(existing_dates)

new_dates = [d for d in forecast_dates if d not in existing_dates.values]
if new_dates:
    new_date_df = pd.DataFrame({'date': new_dates})
    new_date_df['date_key'] = new_date_df['date'].dt.strftime('%Y%m%d').astype(int)
    new_date_df['year'] = new_date_df['date'].dt.year
    new_date_df['quarter'] = new_date_df['date'].dt.quarter
    new_date_df['month_name'] = new_date_df['date'].dt.month_name()
    new_date_df['month_number'] = new_date_df['date'].dt.month
    new_date_df['year_month'] = new_date_df['date'].dt.strftime('%Y-%m')
    new_date_df.to_sql("Dim_Date", dwh_conn, if_exists="append", index=False)

fact_forecast.to_sql("Fact_Product_Forecast", dwh_conn, if_exists="append", index=False)
print(f"Fact_Product_Forecast geladen: {len(fact_forecast)} records")

Fact_Product_Forecast geladen: 3872 records


In [9]:
sdm_conn.close()
dwh_conn.close()
print("ETL Proces voltooid.")

ETL Proces voltooid.
